# LOOM on GSM8K -- A_v1_control against the untuned base

GSM8K is not what LOOM was trained for. It is grade-school arithmetic; LOOM
was trained on physics traces. The question this run answers is **not** "is
LOOM good at maths" but **"did the fine-tune damage the general ability that
was already there"**. A score near the base's is a pass. A score well below it
means the adapter cost something.

Both arms get the identical prompt, greedy decoding and token budget. The only
difference is whether the adapter is attached.

Set **Accelerator: GPU T4 x2** and **Internet: On**.

In [ ]:
!pip -q install -U "transformers>=4.44" accelerate peft "bitsandbytes>=0.46.1" datasets

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, time

OUT_DIR = Path("/kaggle/working/gsm8k")
OUT_DIR.mkdir(parents=True, exist_ok=True)
WORKER = Path("/kaggle/working/gsm8k_worker.py")

# The full official GSM8K test split. A subset would still measure damage, but
# only the complete 1319 produces a number that can be set beside a published
# GSM8K score without an asterisk.
N_PROBLEMS = 1319
# Batch 32 rather than 16: generation is decode-step bound, and the step cost
# grows far slower than the batch, so this buys back most of the time that going
# from 500 to 1319 problems costs. KV cache at this batch is about 1.5 GB
# against the T4's 15, so there is plenty of headroom.
BATCH = 32
MAX_NEW = 640


def _find(pattern, root="/kaggle/input"):
    hits = sorted(Path(root).glob(pattern))
    return hits[0] if hits else None


# The v1 adapter dataset also contains an adapter_config.json, so the search is
# anchored on this arm's own dataset slug rather than on the file name.
_cfg = _find("**/loom-a-v1-control-adapter/**/adapter_config.json") or \
       _find("**/adapter_config.json")
ADAPTER = _cfg.parent if _cfg else None
print("adapter ->", ADAPTER)
assert ADAPTER, "attach the loom-a-v1-control-adapter dataset"
print(json.loads((ADAPTER / "adapter_config.json").read_text())["base_model_name_or_path"])

import torch
NGPU = torch.cuda.device_count()
print(f"GPUs: {NGPU} -> " + ", ".join(torch.cuda.get_device_name(i) for i in range(NGPU)))
assert NGPU >= 1

In [ ]:
WORKER_SRC = r'''"""Score one arm on GSM8K, in its own process on its own GPU.

Same shape as loom_worker.py, and for the same reason: a process exit is the
only way to be sure every byte of VRAM comes back, and one worker per GPU lets
the base and the fine-tune run at the same time instead of one after the other.

    python gsm8k_worker.py --arm BASE --out /kaggle/working/gsm8k
    python gsm8k_worker.py --arm A_v1_control --adapter /path/to/adapter ...

Both arms get the identical prompt, identical decoding (greedy) and identical
token budget. The only difference is whether the adapter is attached.
"""
import argparse
import json
import re
import time
from pathlib import Path

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

# GSM8K gold answers close with "#### <number>".
GOLD = re.compile(r"####\s*([-\d,\.]+)")
# A number, with thousands separators and a trailing full stop allowed.
NUM = re.compile(r"-?\d[\d,]*(?:\.\d+)?")


def gold_answer(text):
    m = GOLD.search(text)
    return float(m.group(1).replace(",", "").rstrip(".")) if m else None


def predicted_answer(text):
    """The model's final number.

    Everything before `</think>` is scratch work full of intermediate numbers,
    so the answer is looked for after it. R1-distill sometimes runs out of
    budget before closing the block; in that case the whole text is used, which
    is the honest fallback -- taking the last number of a truncated trace is
    what a reader would do.
    """
    tail = text.split("</think>")[-1]
    if not NUM.search(tail):
        tail = text
    hits = NUM.findall(tail)
    if not hits:
        return None
    try:
        return float(hits[-1].replace(",", "").rstrip("."))
    except ValueError:
        return None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--arm", required=True)
    ap.add_argument("--adapter", default=None)
    ap.add_argument("--out", required=True)
    ap.add_argument("--n", type=int, default=500)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--max-new", type=int, default=640)
    a = ap.parse_args()

    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    tag = f"[{a.arm}]"
    dev = torch.cuda.get_device_name(0)
    print(f"{tag} {dev} | adapter={a.adapter or 'none'}", flush=True)
    assert torch.cuda.get_device_capability(0)[0] >= 7, "needs a T4 or better"

    tok = AutoTokenizer.from_pretrained(BASE_ID)
    # Decoder-only batched generation must pad on the LEFT, otherwise the pad
    # run sits between the prompt and the first generated token and every
    # sequence in the batch decodes from the wrong position.
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        BASE_ID,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True),
        device_map={"": 0}, torch_dtype=torch.float16)
    if a.adapter:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, a.adapter)
    model.eval()
    print(f"{tag} model ready", flush=True)

    ds = load_dataset("openai/gsm8k", "main", split="test")
    total = len(ds)
    n = min(a.n, total)
    # A fixed shuffle, so both arms see the same problems in the same order and
    # a subset is not just the front of the file.
    ds = ds.shuffle(seed=0).select(range(n))
    print(f"{tag} {n} of {total} GSM8K test problems", flush=True)

    prompts = [tok.apply_chat_template(
        [{"role": "user", "content": r["question"]}],
        add_generation_prompt=True, tokenize=False) for r in ds]
    golds = [gold_answer(r["answer"]) for r in ds]

    rows, correct, truncated, t0 = [], 0, 0, time.time()
    for i in range(0, n, a.batch):
        batch = prompts[i:i + a.batch]
        enc = tok(batch, return_tensors="pt", padding=True,
                  add_special_tokens=False).to("cuda")
        with torch.inference_mode():
            gen = model.generate(**enc, max_new_tokens=a.max_new,
                                 do_sample=False, temperature=None, top_p=None,
                                 pad_token_id=tok.pad_token_id)
        outs = tok.batch_decode(gen[:, enc["input_ids"].shape[1]:],
                                skip_special_tokens=True)
        for j, text in enumerate(outs):
            k = i + j
            pred, gold = predicted_answer(text), golds[k]
            ok = pred is not None and gold is not None and abs(pred - gold) < 1e-4
            correct += ok
            closed = "</think>" in text
            truncated += not closed
            rows.append({"i": k, "gold": gold, "pred": pred, "correct": bool(ok),
                         "closed_think": closed, "chars": len(text),
                         "output": text if k < 20 else None})
        done = min(i + a.batch, n)
        el = time.time() - t0
        print(f"{tag} {done:4d}/{n}  acc {correct / done:.3f}  "
              f"{el / done:.1f}s/problem  eta {(n - done) * el / done / 60:.0f}m",
              flush=True)

    res = {"arm": a.arm, "adapter": a.adapter, "n": n,
           "accuracy": round(correct / n, 4), "correct": correct,
           "closed_think_rate": round(1 - truncated / n, 4),
           "max_new_tokens": a.max_new, "minutes": round((time.time() - t0) / 60, 1),
           "per_problem": rows}
    (out / f"gsm8k_{a.arm}.json").write_text(json.dumps(res, indent=2),
                                             encoding="utf-8")
    print(f"{tag} DONE accuracy {res['accuracy']} "
          f"({correct}/{n}) in {res['minutes']} min", flush=True)


if __name__ == "__main__":
    main()
'''
WORKER.write_text(WORKER_SRC, encoding="utf-8")
print(f"wrote {WORKER} ({len(WORKER_SRC):,} bytes)")

In [ ]:
ARMS = [("BASE", None), ("A_v1_control", str(ADAPTER))]


def launch(arm, adapter, gpu):
    cmd = [sys.executable, str(WORKER), "--arm", arm, "--out", str(OUT_DIR),
           "--n", str(N_PROBLEMS), "--batch", str(BATCH), "--max-new", str(MAX_NEW)]
    if adapter:
        cmd += ["--adapter", adapter]
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu),
           "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
           "HF_HUB_ENABLE_HF_TRANSFER": "0"}
    log = (OUT_DIR / f"log_{arm}.txt").open("w", encoding="utf-8")
    print(f"  -> {arm} starting on GPU {gpu}", flush=True)
    return {"arm": arm, "log": log, "pos": 0,
            "proc": subprocess.Popen(cmd, env=env, stdout=log,
                                     stderr=subprocess.STDOUT, text=True)}


def drain(job):
    p = OUT_DIR / f"log_{job['arm']}.txt"
    if not p.exists():
        return
    txt = p.read_text(encoding="utf-8", errors="replace")
    new, job["pos"] = txt[job["pos"]:], len(txt)
    for line in new.splitlines():
        if line.strip():
            print(line, flush=True)


queue = list(ARMS)
running, t0 = [], time.time()
while queue or running:
    while queue and len(running) < NGPU:
        arm, ad = queue.pop(0)
        running.append(launch(arm, ad, len(running) % NGPU))
    time.sleep(30)
    for job in list(running):
        drain(job)
        if job["proc"].poll() is not None:
            job["log"].close(); drain(job)
            print(f"== {job['arm']} exited rc={job['proc'].returncode} "
                  f"({(time.time() - t0) / 60:.0f} min)", flush=True)
            running.remove(job)

print(f"\nboth arms finished in {(time.time() - t0) / 60:.0f} min")

## Compare

The number that matters is the **gap**, not either score on its own.

In [ ]:
res = {}
for arm, _ in ARMS:
    f = OUT_DIR / f"gsm8k_{arm}.json"
    if f.exists():
        res[arm] = json.loads(f.read_text(encoding="utf-8"))
    else:
        print(f"!! {arm} produced no result -- see log_{arm}.txt")

hdr = f"{'arm':16s} {'accuracy':>9s} {'correct':>9s} {'closed':>7s} {'min':>6s}"
print(hdr); print("-" * len(hdr))
for arm, r in res.items():
    print(f"{arm:16s} {r['accuracy']:9.3f} {r['correct']:>4d}/{r['n']:<4d} "
          f"{r['closed_think_rate']:7.2f} {r['minutes']:6.1f}")

if len(res) == 2:
    b, a = res["BASE"], res["A_v1_control"]
    gap = a["accuracy"] - b["accuracy"]
    # Two independent proportions on the same n; the standard error of the
    # difference is what decides whether a gap means anything at all.
    import math
    se = math.sqrt(sum(r["accuracy"] * (1 - r["accuracy"]) / r["n"] for r in (b, a)))
    print(f"\ngap {gap * 100:+.1f} points, standard error {se * 100:.1f} points")
    if abs(gap) <= 2 * se:
        print("-> inside noise: no measurable damage to general ability")
    elif gap < 0:
        print("-> the fine-tune costs general ability by more than noise")
    else:
        print("-> the fine-tune helps here by more than noise")

Path("/kaggle/working/gsm8k_report.json").write_text(json.dumps({
    "created": time.strftime("%Y-%m-%d %H:%M"),
    "n": N_PROBLEMS, "max_new_tokens": MAX_NEW,
    "results": {k: {kk: vv for kk, vv in v.items() if kk != "per_problem"}
                for k, v in res.items()},
    "per_problem": {k: v["per_problem"] for k, v in res.items()},
}, indent=2), encoding="utf-8")
print("\nwrote gsm8k_report.json")